In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv
/kaggle/input/datasets/kanchandalal123/mcq-ranking-train-dataset-2/ranking_train (1).csv


# Smart MCQ Solver Challenge - DeBERTa Optimization

## Objective

In the previous notebook, we established a strong DeBERTa baseline for the Smart MCQ Solver Challenge.

This notebook focuses on improving the baseline through systematic hyperparameter optimization and better training strategies.

The objectives are:

- Improve MAP@3 performance
- Reduce overfitting
- Automatically select the best checkpoint
- Track experiments using Weights & Biases (WandB)
- Save the best trained model
- Generate a competition submission file

Unlike the baseline notebook, this notebook applies training optimization techniques while keeping the model architecture unchanged.

In [2]:
import os
import gc
import random
import warnings

import numpy as np
import pandas as pd

import torch

from datasets import Dataset

from sklearn.metrics import (
    accuracy_score,
    f1_score
)

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    EarlyStoppingCallback
)

import wandb

warnings.filterwarnings("ignore")

In [3]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

os.environ["PYTHONHASHSEED"] = str(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [4]:
os.environ["WANDB_API_KEY"] = "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"

wandb.init(

    project="24f1002360-t22026",

    name="deberta_optimization_v1",

    config={

        "model":"microsoft/deberta-v3-base",

        "epochs":25,

        "batch_size":8,

        "learning_rate":1e-5,

        "scheduler":"cosine",

        "max_length":256,

        "weight_decay":0.01,

        "early_stopping":True

    }
)

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: 24f1002360 (24f1002360-indian-institute-of-technology-madras) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [5]:
df = pd.read_csv(
    "/kaggle/input/datasets/kanchandalal123/mcq-ranking-train-dataset-2/ranking_train (1).csv"
)

print(df.shape)

df.head()

(10000, 8)


,id,question,option_label,option_text,label,text,text_length,fold
0,1,Pick the best possible answer: What is Martin ...,A,Martin Heidegger believes that humans exist wi...,0,Question: Pick the best possible answer: What ...,474,3
1,1,Pick the best possible answer: What is Martin ...,B,Martin Heidegger believes that humans do not e...,1,Question: Pick the best possible answer: What ...,425,3
2,1,Pick the best possible answer: What is Martin ...,C,Martin Heidegger does not believe in the exist...,0,Question: Pick the best possible answer: What ...,389,3
3,1,Pick the best possible answer: What is Martin ...,D,Martin Heidegger believes that the relationshi...,0,Question: Pick the best possible answer: What ...,369,3
4,1,Pick the best possible answer: What is Martin ...,E,Martin Heidegger believes that time is an illu...,0,Question: Pick the best possible answer: What ...,358,3


In [6]:
FOLD = 0

train_df = (

    df[df.fold != FOLD]

    .reset_index(drop=True)

)

valid_df = (

    df[df.fold == FOLD]

    .reset_index(drop=True)

)

print(train_df.shape)

print(valid_df.shape)

(8000, 8)
(2000, 8)


In [7]:
MODEL_NAME = "microsoft/deberta-v3-base"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

In [8]:
MAX_LENGTH = 256

def tokenize(example):

    encoded = tokenizer(

        example["text"],

        truncation=True,

        max_length=MAX_LENGTH

    )

    encoded.pop("token_type_ids", None)

    return encoded

In [9]:
train_ds = Dataset.from_pandas(
    train_df[["text", "label"]]
)

valid_ds = Dataset.from_pandas(
    valid_df[["text", "label"]]
)

In [10]:
train_ds = train_ds.map(
    tokenize,
    batched=True
)

valid_ds = valid_ds.map(
    tokenize,
    batched=True
)

Map:   0%|          | 0/8000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [11]:
train_ds = train_ds.remove_columns(["text"])

valid_ds = valid_ds.remove_columns(["text"])

train_ds.set_format("torch")

valid_ds.set_format("torch")

In [12]:
model = AutoModelForSequenceClassification.from_pretrained(

    MODEL_NAME,

    num_labels=2,

    torch_dtype=torch.float32

)

print(next(model.parameters()).dtype)

`torch_dtype` is deprecated! Use `dtype` instead!


pytorch_model.bin:   0%|          | 0.00/371M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/371M [00:00<?, ?B/s]

DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
classifier.weight                       | MISSING    | 
classifier.bias                         | MISSING    | 
pooler.dense.weight                     | MISSING    | 
pooler.dense.bias        

torch.float32


In [13]:
def apk(actual, predicted, k=3):

    if actual in predicted[:k]:

        return 1.0 / (
            predicted.index(actual) + 1
        )

    return 0.0


def mapk(actuals, predictions, k=3):

    return np.mean(

        [

            apk(a, p, k)

            for a, p in zip(actuals, predictions)

        ]

    )

In [14]:
valid_metadata = valid_df[
    [
        "id",
        "option_label",
        "label"
    ]
].reset_index(drop=True)

In [15]:
def compute_metrics(eval_pred):

    logits, labels = eval_pred

    probs = torch.softmax(

        torch.tensor(logits),

        dim=1

    )[:,1].numpy()

    temp = valid_metadata.copy()

    temp["score"] = probs

    predictions = []

    actuals = []

    for _, grp in temp.groupby("id"):

        grp = grp.sort_values(

            "score",

            ascending=False

        )

        predictions.append(

            grp["option_label"].tolist()

        )

        actuals.append(

            grp.loc[
                grp["label"] == 1,
                "option_label"
            ].values[0]
        )

    preds = np.argmax(

        logits,

        axis=1

    )

    return {

        "accuracy": accuracy_score(
            labels,
            preds
        ),

        "f1": f1_score(
            labels,
            preds
        ),

        "map3": mapk(
            actuals,
            predictions
        )

    }

In [16]:
early_stop = EarlyStoppingCallback(

    early_stopping_patience=2,

    early_stopping_threshold=0.0005

)

In [19]:
training_args = TrainingArguments(

    output_dir="deberta_optimization",

    learning_rate=1e-5,

    per_device_train_batch_size=8,

    per_device_eval_batch_size=8,

    num_train_epochs=25,

    weight_decay=0.01,

    warmup_steps=0.10,

    lr_scheduler_type="cosine",

    eval_strategy="epoch",

    save_strategy="epoch",

    logging_strategy="steps",

    logging_steps=50,

    save_total_limit=2,

    load_best_model_at_end=True,

    metric_for_best_model="map3",

    greater_is_better=True,

    report_to="wandb",

    fp16=False,

    bf16=False,

    seed=SEED
)

In [20]:
data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer
)

In [21]:
trainer = Trainer(

    model=model,

    args=training_args,

    train_dataset=train_ds,

    eval_dataset=valid_ds,

    processing_class=tokenizer,

    data_collator=data_collator,

    compute_metrics=compute_metrics,

    callbacks=[early_stop]
)

In [22]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 2, 'bos_token_id': 1}.


Epoch,Training Loss,Validation Loss,Accuracy,F1,Map3
1,0.983373,0.997158,0.800000,0.000000,0.579167
2,1.016668,0.967160,0.800000,0.000000,0.758750
3,0.677164,0.477427,0.915000,0.752907,0.969583
4,0.362957,0.200519,0.972500,0.933009,0.993750
5,0.172956,0.081028,0.989500,0.973451,0.998750
6,0.099775,0.037688,0.996000,0.989899,1.000000
7,0.047686,0.017842,0.997500,0.993726,1.000000
8,0.031235,0.010685,0.997500,0.993773,1.000000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['deberta.embeddings.LayerNorm.weight', 'deberta.embeddings.LayerNorm.bias', 'deberta.encoder.layer.0.attention.output.LayerNorm.weight', 'deberta.encoder.layer.0.attention.output.LayerNorm.bias', 'deberta.encoder.layer.0.output.LayerNorm.weight', 'deberta.encoder.layer.0.output.LayerNorm.bias', 'deberta.encoder.layer.1.attention.output.LayerNorm.weight', 'deberta.encoder.layer.1.attention.output.LayerNorm.bias', 'deberta.encoder.layer.1.output.LayerNorm.weight', 'deberta.encoder.layer.1.output.LayerNorm.bias', 'deberta.encoder.layer.2.attention.output.LayerNorm.weight', 'deberta.encoder.layer.2.attention.output.LayerNorm.bias', 'deberta.encoder.layer.2.output.LayerNorm.weight', 'deberta.encoder.layer.2.output.LayerNorm.bias', 'deberta.encoder.layer.3.attention.output.LayerNorm.weight', 'deberta.encoder.layer.3.attention.output.LayerNorm.bias', 'deberta.encoder.layer.3.output.LayerNorm.weight', 'deberta.encoder.layer.3.output.Laye

TrainOutput(global_step=4000, training_loss=0.46433728309720756, metrics={'train_runtime': 1660.5354, 'train_samples_per_second': 120.443, 'train_steps_per_second': 7.528, 'total_flos': 3514634729507328.0, 'train_loss': 0.46433728309720756, 'epoch': 8.0})

In [23]:
results = trainer.evaluate()

results

{'eval_loss': 0.037355467677116394,
 'eval_accuracy': 0.996,
 'eval_f1': 0.98989898989899,
 'eval_map3': 1.0,
 'eval_runtime': 16.3688,
 'eval_samples_per_second': 122.184,
 'eval_steps_per_second': 7.637,
 'epoch': 8.0}

In [24]:
metrics = pd.DataFrame([results])

metrics.to_csv(
    "deberta_optimization_metrics.csv",
    index=False
)

metrics

,eval_loss,eval_accuracy,eval_f1,eval_map3,eval_runtime,eval_samples_per_second,eval_steps_per_second,epoch
0,0.037355,0.996,0.989899,1.0,16.3688,122.184,7.637,8.0


In [25]:
SAVE_PATH = "/kaggle/working/deberta_optimized_model"

trainer.save_model(SAVE_PATH)

tokenizer.save_pretrained(SAVE_PATH)

print("Model Saved Successfully")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model Saved Successfully


In [26]:
gc.collect()

torch.cuda.empty_cache()

In [27]:
test = pd.read_csv(
    "/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv"
)

print(test.shape)

test.head()

(500, 7)


,id,prompt,A,B,C,D,E
0,1,Pick the best possible answer: What is the rel...,"For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p..."
1,2,"What is the estimated redshift of CEERS-93316,...","Approximately z = 6.0, corresponding to 1 bill...","Approximately z = 16.7, corresponding to 235.8...","Approximately z = 3.0, corresponding to 5 bill...","Approximately z = 10.0, corresponding to 13 bi...","Approximately z = 13.0, corresponding to 30 bi..."
2,3,Pick the best possible answer: What is the rea...,The sun appears yellowish due to a reflection ...,"The longer wavelengths of light, such as red a...",The sun appears yellowish due to the scatterin...,The sun emits a yellow light due to its own sp...,The atmosphere absorbs the shorter wavelengths...
3,4,What is the significance of the redshift-dista...,Observations of the redshift-distance relation...,Observations of the redshift-distance relation...,Observations of the redshift-distance relation...,Observations of the redshift-distance relation...,Observations of the redshift-distance relation...
4,5,What is the Landau-Lifshitz-Gilbert equation u...,The Landau-Lifshitz-Gilbert equation is a diff...,The Landau-Lifshitz-Gilbert equation is a diff...,The Landau-Lifshitz-Gilbert equation is a diff...,The Landau-Lifshitz-Gilbert equation is a diff...,The Landau-Lifshitz-Gilbert equation is a diff...


In [28]:
def create_test_ranking(df):

    rows = []

    for _, row in df.iterrows():

        for option in ["A", "B", "C", "D", "E"]:

            rows.append({

                "id": row["id"],

                "question": row["prompt"],

                "option_label": option,

                "option_text": row[option]

            })

    return pd.DataFrame(rows)

In [29]:
ranking_test = create_test_ranking(test)

ranking_test["text"] = (

    "Question: "

    + ranking_test["question"]

    + " [SEP] Option: "

    + ranking_test["option_text"]

)

ranking_test.head()

,id,question,option_label,option_text,text
0,1,Pick the best possible answer: What is the rel...,A,"For every eigenstate of one Hamiltonian, its p...",Question: Pick the best possible answer: What ...
1,1,Pick the best possible answer: What is the rel...,B,"For every eigenstate of one Hamiltonian, its p...",Question: Pick the best possible answer: What ...
2,1,Pick the best possible answer: What is the rel...,C,"For every eigenstate of one Hamiltonian, its p...",Question: Pick the best possible answer: What ...
3,1,Pick the best possible answer: What is the rel...,D,"For every eigenstate of one Hamiltonian, its p...",Question: Pick the best possible answer: What ...
4,1,Pick the best possible answer: What is the rel...,E,"For every eigenstate of one Hamiltonian, its p...",Question: Pick the best possible answer: What ...


In [31]:
test = pd.read_csv(
    "/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv"
)

rows = []

for _, row in test.iterrows():

    for option in ["A", "B", "C", "D", "E"]:

        rows.append({

            "id": row["id"],

            "option_label": option,

            "text":
                "Question: "
                + row["prompt"]
                + " [SEP] Option: "
                + row[option]

        })

ranking_test = pd.DataFrame(rows)

ranking_test.head()

,id,option_label,text
0,1,A,Question: Pick the best possible answer: What ...
1,1,B,Question: Pick the best possible answer: What ...
2,1,C,Question: Pick the best possible answer: What ...
3,1,D,Question: Pick the best possible answer: What ...
4,1,E,Question: Pick the best possible answer: What ...


In [32]:
from datasets import Dataset

test_ds = Dataset.from_pandas(ranking_test)

test_ds = test_ds.map(

    lambda x: tokenizer(

        x["text"],

        truncation=True,

        max_length=256

    ),

    batched=True

)

Map:   0%|          | 0/2500 [00:00<?, ? examples/s]

In [33]:
test_ds.set_format(

    type="torch",

    columns=[

        "input_ids",

        "attention_mask"

    ]
)

In [36]:
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer,
    return_tensors="pt"
)

test_loader = DataLoader(
    test_ds,
    batch_size=32,
    shuffle=False,
    collate_fn=data_collator
)

In [37]:
model.eval()

all_probs = []

with torch.no_grad():

    for batch in test_loader:

        batch = {

            k: v.to(model.device)

            for k, v in batch.items()

        }

        outputs = model(

            input_ids=batch["input_ids"],

            attention_mask=batch["attention_mask"]

        )

        probs = F.softmax(

            outputs.logits,

            dim=1

        )[:,1]

        all_probs.extend(

            probs.cpu().numpy()

        )

In [38]:
ranking_test["score"] = all_probs

ranking_test.head()

,id,option_label,text,score
0,1,A,Question: Pick the best possible answer: What ...,0.999836
1,1,B,Question: Pick the best possible answer: What ...,0.000043
2,1,C,Question: Pick the best possible answer: What ...,0.000035
3,1,D,Question: Pick the best possible answer: What ...,0.000038
4,1,E,Question: Pick the best possible answer: What ...,0.000033


In [39]:
submission = (

    ranking_test

    .sort_values(

        ["id","score"],

        ascending=[True,False]

    )

    .groupby("id")["option_label"]

    .apply(

        lambda x:

        " ".join(x.head(3))

    )

    .reset_index()

)

submission.columns = [

    "ID",

    "Prediction"

]

submission.head()

,ID,Prediction
0,1,A B D
1,2,B E C
2,3,B D E
3,4,E C D
4,5,C D B


In [40]:
submission.to_csv(

    "submission.csv",

    index=False

)

print(submission.head())

print()

print("Submission Saved!")

   ID Prediction
0   1      A B D
1   2      B E C
2   3      B D E
3   4      E C D
4   5      C D B

Submission Saved!


In [41]:
import shutil

shutil.make_archive(
    "/kaggle/working/deberta_optimized_model",
    "zip",
    "/kaggle/working/deberta_optimized_model"
)

print("Model zipped successfully!")


Model zipped successfully!
